In [4]:
# ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
# A suitable version of pyarrow or fastparquet is required for parquet support.
# Trying to import the above resulted in these errors:
#  - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
#  - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.
# Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings... 
!pip install pyarrow
!pip install fastparquet

   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
   --- ------------------------------------ 2.6/27.5 MB 16.7 MB/s eta 0:00:02
   --------- ------------------------------ 6.3/27.5 MB 16.8 MB/s eta 0:00:02
   ------------- -------------------------- 9.2/27.5 MB 15.4 MB/s eta 0:00:02
   ----------------- ---------------------- 12.1/27.5 MB 15.1 MB/s eta 0:00:02
   --------------------- ------------------ 14.9/27.5 MB 14.7 MB/s eta 0:00:01
   ------------------------- -------------- 17.8/27.5 MB 14.6 MB/s eta 0:00:01
   ----------------------------- ---------- 20.4/27.5 MB 14.2 MB/s eta 0:00:01
   --------------------------------- ------ 23.3/27.5 MB 14.1 MB/s eta 0:00:01
   ------------------------------------- -- 26.0/27.5 MB 13.8 MB/s eta 0:00:01
   ---------------------------------------  27.3/27.5 MB 13.8 MB/s eta 0:00:01
   ---------------------------------------- 27.5/27.5 MB 13.2 MB/s  0:00:02



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/669.8 kB ? eta -:--:--
   ---------------------------------------- 669.8/669.8 kB 13.1 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 13.2 MB/s  0:00:00

   -------------------- ------------------- 1/2 [fastparquet]
   -------------------- ------------------- 1/2 [fastparquet]
   ---------------------------------------- 2/2 [fastparquet]




[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import io
import requests
import pandas as pd

def load_data(*args, **kwargs):
    year = kwargs.get('year', 2025)
    month = kwargs.get('month', '01')

    url = f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month}.parquet'

    # Some institutional networks block generic urllib traffic and return HTTP 403.
    headers = {
        'User-Agent': kwargs.get(
            'user_agent',
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
            '(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36'
        )
    }

    response = requests.get(url, headers=headers, timeout=60)
    response.raise_for_status()

    parquet_buffer = io.BytesIO(response.content)

    engine = kwargs.get('engine', 'fastparquet')
    try:
        datos_crudos = pd.read_parquet(parquet_buffer, engine=engine)
    except Exception:
        parquet_buffer.seek(0)
        datos_crudos = pd.read_parquet(parquet_buffer, engine='fastparquet')

    return datos_crudos

data = load_data(year=2025, month='01')
data.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [2]:
data.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee'],
      dtype='object')

In [3]:
def transform(data, *args, **kwargs):
    data.columns = [col.lower() for col in data.columns]
    
    data.rename(columns={
        'vendorid': 'vendor_id',
        'ratecodeid': 'rate_code_id',
        'pulocationid': 'pu_location_id',
        'dolocationid': 'do_location_id'
    }, inplace = True)

    return data

data = transform(data)
data.columns

Index(['vendor_id', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'rate_code_id',
       'store_and_fwd_flag', 'pu_location_id', 'do_location_id',
       'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount',
       'tolls_amount', 'improvement_surcharge', 'total_amount',
       'congestion_surcharge', 'airport_fee', 'cbd_congestion_fee'],
      dtype='object')

In [4]:
data.head()

,vendor_id,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pu_location_id,do_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [20]:
rows = data.shape[0]
chunksize = 500000

print(f'Total rows: {rows}')
print(f'Chunk size: {chunksize}')

print("Replace")

print(f'Chunk {0//chunksize + 1}: {chunksize} rows')

print("Append")

for i in range(chunksize, rows, chunksize):
    chunk = data.iloc[i:i+chunksize]
    print(f'Chunk {i//chunksize + 1}: {chunk.shape[0]} rows')

Total rows: 12741035
Chunk size: 500000
Replace
Chunk 1: 500000 rows
Append
Chunk 2: 500000 rows
Chunk 3: 500000 rows
Chunk 4: 500000 rows
Chunk 5: 500000 rows
Chunk 6: 500000 rows
Chunk 7: 500000 rows
Chunk 8: 500000 rows
Chunk 9: 500000 rows
Chunk 10: 500000 rows
Chunk 11: 500000 rows
Chunk 12: 500000 rows
Chunk 13: 500000 rows
Chunk 14: 500000 rows
Chunk 15: 500000 rows
Chunk 16: 500000 rows
Chunk 17: 500000 rows
Chunk 18: 500000 rows
Chunk 19: 500000 rows
Chunk 20: 500000 rows
Chunk 21: 500000 rows
Chunk 22: 500000 rows
Chunk 23: 500000 rows
Chunk 24: 500000 rows
Chunk 25: 500000 rows
Chunk 26: 241035 rows


In [5]:
for col in data.columns:
    null_count = data[col].isnull().sum()
    print(f'Null values in column {col}: {null_count}')

Null values in column vendor_id: 0
Null values in column tpep_pickup_datetime: 0
Null values in column tpep_dropoff_datetime: 0
Null values in column passenger_count: 540149
Null values in column trip_distance: 0
Null values in column rate_code_id: 540149
Null values in column store_and_fwd_flag: 540149
Null values in column pu_location_id: 0
Null values in column do_location_id: 0
Null values in column payment_type: 0
Null values in column fare_amount: 0
Null values in column extra: 0
Null values in column mta_tax: 0
Null values in column tip_amount: 0
Null values in column tolls_amount: 0
Null values in column improvement_surcharge: 0
Null values in column total_amount: 0
Null values in column congestion_surcharge: 540149
Null values in column airport_fee: 540149
Null values in column cbd_congestion_fee: 0


In [15]:
data[data["passenger_count"].isna()]["vendor_id"].value_counts()

vendor_id
2    451456
1     88204
6       489
Name: count, dtype: int64

In [17]:
data[data["passenger_count"] == 0]

,vendor_id,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pu_location_id,do_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee
6,1,2025-01-01 00:14:47,2025-01-01 00:16:15,0.0,0.4,1.0,N,170,170,1,4.4,3.50,0.5,2.35,0.0,1.0,11.75,2.5,0.0,0.00
7,1,2025-01-01 00:39:27,2025-01-01 00:51:51,0.0,1.6,1.0,N,234,148,1,12.1,3.50,0.5,2.00,0.0,1.0,19.10,2.5,0.0,0.00
8,1,2025-01-01 00:53:43,2025-01-01 01:13:23,0.0,2.8,1.0,N,148,170,1,19.1,3.50,0.5,3.00,0.0,1.0,27.10,2.5,0.0,0.00
94,1,2025-01-01 00:11:27,2025-01-01 00:16:58,0.0,0.7,1.0,N,144,211,1,7.2,3.50,0.5,0.00,0.0,1.0,12.20,2.5,0.0,0.00
95,1,2025-01-01 00:19:30,2025-01-01 00:27:25,0.0,1.0,1.0,N,211,158,1,9.3,3.50,0.5,2.85,0.0,1.0,17.15,2.5,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2933984,1,2025-01-31 23:20:45,2025-01-31 23:45:28,0.0,3.9,1.0,N,144,50,1,25.4,4.25,0.5,2.50,0.0,1.0,33.65,2.5,0.0,0.75
2934049,1,2025-01-31 23:54:49,2025-02-01 00:09:35,0.0,6.4,1.0,N,263,232,1,28.2,4.25,0.5,2.00,0.0,1.0,35.95,2.5,0.0,0.75
2934589,1,2025-01-31 23:16:04,2025-01-31 23:36:38,0.0,2.4,1.0,N,68,114,1,15.6,4.25,0.5,4.25,0.0,1.0,25.60,2.5,0.0,0.75
2934590,1,2025-01-31 23:57:51,2025-02-01 00:08:06,0.0,1.6,1.0,N,142,236,1,10.7,3.50,0.5,3.10,0.0,1.0,18.80,2.5,0.0,0.00


In [76]:
# Data Quality Analysis (aligned with NYC TLC data dictionary)

valid_vendor_ids = {1, 2, 6, 7}
valid_rate_codes = {1, 2, 3, 4, 5, 6, 99}
valid_payment_types = {0, 1, 2, 3, 4, 5, 6}
valid_store_flags = {"Y", "N"}

print("-" * 80)
completitud = pd.DataFrame({
    'Column': data.columns,
    'Missing_Count': [data[col].isnull().sum() for col in data.columns],
    'Missing_Percentage': [(data[col].isnull().sum() / len(data)) * 100 for col in data.columns]
})
completitud = completitud.sort_values('Missing_Percentage', ascending=False)
print(completitud.to_string(index=False))

print("-" * 80)

# Non-negative checks
non_negative_cols = [
    'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax',
    'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount',
    'congestion_surcharge', 'airport_fee', 'cbd_congestion_fee'
 ]

for col in non_negative_cols:
    if col in data.columns:
        negatives = (data[col] < 0).sum()
        print(f"{col} < 0: {negatives}")

pickup_after_dropoff = (
    data['tpep_pickup_datetime'] > data['tpep_dropoff_datetime']
).sum()
print(f"\nPickup datetime after dropoff datetime: {pickup_after_dropoff}")

invalid_vendor = (~data['vendor_id'].isin(valid_vendor_ids) & data['vendor_id'].notna()).sum()
invalid_rate = (~data['rate_code_id'].isin(valid_rate_codes) & data['rate_code_id'].notna()).sum()
invalid_payment = (~data['payment_type'].isin(valid_payment_types) & data['payment_type'].notna()).sum()

store_flag = data['store_and_fwd_flag'].astype('string').str.strip().str.upper()
invalid_store = (~store_flag.isin(valid_store_flags) & store_flag.notna()).sum()

print(f"Invalid vendor_id values {sorted(valid_vendor_ids)}: {invalid_vendor}")
print(f"Invalid rate_code_id values {sorted(valid_rate_codes)}: {invalid_rate}")
print(f"Invalid payment_type values {sorted(valid_payment_types)}: {invalid_payment}")
print(f"Invalid store_and_fwd_flag values {sorted(valid_store_flags)}: {invalid_store}")

print("-" * 80)
print(f"\nPassenger count range: {data['passenger_count'].min()} to {data['passenger_count'].max()}")
print(f"Trip distance range: {data['trip_distance'].min()} to {data['trip_distance'].max()}")

--------------------------------------------------------------------------------
               Column  Missing_Count  Missing_Percentage
      passenger_count         540149           15.542845
 congestion_surcharge         540149           15.542845
   store_and_fwd_flag         540149           15.542845
         rate_code_id         540149           15.542845
          airport_fee         540149           15.542845
tpep_dropoff_datetime              0            0.000000
            vendor_id              0            0.000000
 tpep_pickup_datetime              0            0.000000
       do_location_id              0            0.000000
         payment_type              0            0.000000
        trip_distance              0            0.000000
       pu_location_id              0            0.000000
                extra              0            0.000000
          fare_amount              0            0.000000
              mta_tax              0            0.000000
       

In [78]:
# Se valida que todos los nulls estén en las mismas filas, lo que sugiere que no hay columnas con nulls independientes

null_rows = data[data.isnull().any(axis=1)].index

print(f"Number of rows with any nulls: {len(null_rows)}")

Number of rows with any nulls: 540149


In [231]:
data.iloc[null_rows]

,vendor_id,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pu_location_id,do_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee
2935077,2,2025-01-01 00:39:17,2025-01-01 01:10:42,NaN,6.75,NaN,None,48,42,0,2.32,0.0,0.5,0.0,0.0,1.0,14.90,NaN,NaN,0.00
2935078,2,2025-01-01 00:33:40,2025-01-01 01:10:34,NaN,3.88,NaN,None,4,48,0,6.02,0.0,0.5,0.0,0.0,1.0,10.02,NaN,NaN,0.00
2935079,1,2025-01-01 00:09:32,2025-01-01 00:12:38,NaN,0.00,NaN,None,158,158,0,22.72,0.0,0.5,0.0,0.0,1.0,26.72,NaN,NaN,0.00
2935080,2,2025-01-01 00:21:42,2025-01-01 00:36:29,NaN,3.55,NaN,None,137,211,0,-2.35,0.0,0.5,0.0,0.0,1.0,5.80,NaN,NaN,0.00
2935081,2,2025-01-01 00:56:04,2025-01-01 01:06:11,NaN,1.55,NaN,None,140,233,0,5.06,0.0,0.5,0.0,0.0,1.0,9.06,NaN,NaN,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3475221,2,2025-01-31 23:01:48,2025-01-31 23:16:29,NaN,3.35,NaN,None,79,237,0,15.85,0.0,0.5,0.0,0.0,1.0,20.60,NaN,NaN,0.75
3475222,2,2025-01-31 23:50:29,2025-02-01 00:17:27,NaN,8.73,NaN,None,161,116,0,28.14,0.0,0.5,0.0,0.0,1.0,32.89,NaN,NaN,0.75
3475223,2,2025-01-31 23:26:59,2025-01-31 23:43:01,NaN,2.64,NaN,None,144,246,0,14.91,0.0,0.5,0.0,0.0,1.0,19.66,NaN,NaN,0.75
3475224,2,2025-01-31 23:14:34,2025-01-31 23:34:52,NaN,3.16,NaN,None,142,107,0,17.55,0.0,0.5,0.0,0.0,1.0,22.30,NaN,NaN,0.75


In [ ]:
data.iloc[null_rows]["payment_type"].value_counts(), data.iloc[~null_rows]["payment_type"].value_counts()

# Todos los nulls corresponden a "payment_type" = 0 (Flex Fare trip), se decide mantenerlos como valores null

(payment_type
 0    540149
 Name: count, dtype: int64,
 payment_type
 1    426992
 2     91136
 4     16708
 3      5312
 5         1
 Name: count, dtype: int64)

In [98]:
# Se identifica de dónde provienen fare_amount < 0

print("Negative values in fare_amount by vendor_id:")
print(data[data["fare_amount"] < 0]["vendor_id"].value_counts())
print(data[data["fare_amount"] >= 0]["vendor_id"].value_counts())
print("Solamente VendorID 2 tiene valores negativos en fare_amount, pero no todos los viajes de VendorID 2 tienen valores negativos en fare_amount, se decide mantenerlos como valores negativos")

print("\nNegative values in fare_amount by mta_tax:")
print(data[data["fare_amount"] < 0]["mta_tax"].value_counts())

Negative values in fare_amount by vendor_id:
vendor_id
2    144118
Name: count, dtype: int64
vendor_id
2    2575742
1     753671
7       1206
6        489
Name: count, dtype: int64
Solamente VendorID 2 tiene valores negativos en fare_amount, pero no todos los viajes de VendorID 2 tienen valores negativos en fare_amount, se decide mantenerlos como valores negativos

Negative values in fare_amount by mta_tax:
mta_tax
 0.5    84799
-0.5    56924
 0.0     2395
Name: count, dtype: int64


In [183]:
data[(data["tolls_amount"] < 0)]

,vendor_id,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pu_location_id,do_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee
822,2,2025-01-01 00:49:36,2025-01-01 02:11:46,4.0,24.70,1.0,N,68,82,3,-111.5,-1.0,-0.5,0.0,-6.94,-1.0,-123.44,-2.5,0.00,0.00
1061,2,2025-01-01 00:05:15,2025-01-01 00:23:39,1.0,7.79,1.0,N,138,24,4,-32.4,-6.0,-0.5,0.0,-6.94,-1.0,-48.59,0.0,-1.75,0.00
3425,2,2025-01-01 00:33:51,2025-01-01 01:07:12,2.0,19.60,2.0,N,132,238,2,-70.0,0.0,-0.5,0.0,-6.94,-1.0,-80.19,0.0,-1.75,0.00
3579,2,2025-01-01 00:38:09,2025-01-01 01:16:04,1.0,19.66,1.0,N,234,210,4,-76.5,-1.0,-0.5,0.0,-6.94,-1.0,-88.44,-2.5,0.00,0.00
4041,2,2025-01-01 00:16:59,2025-01-01 00:47:34,4.0,19.66,1.0,N,48,80,2,-74.4,-1.0,-0.5,0.0,-6.94,-1.0,-86.34,-2.5,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2933506,2,2025-01-31 23:55:27,2025-02-01 00:10:01,1.0,8.37,1.0,N,70,141,4,-33.1,-1.0,-0.5,0.0,-6.94,-1.0,-47.54,-2.5,-1.75,-0.75
2933777,2,2025-01-31 23:57:21,2025-02-01 00:09:32,1.0,8.04,1.0,N,70,248,4,-31.7,-6.0,-0.5,0.0,-6.94,-1.0,-47.89,0.0,-1.75,0.00
2934041,2,2025-01-31 23:15:38,2025-01-31 23:42:56,2.0,17.74,2.0,N,132,75,4,-70.0,0.0,-0.5,0.0,-6.94,-1.0,-80.19,0.0,-1.75,0.00
2934148,2,2025-01-31 23:15:44,2025-01-31 23:48:13,2.0,17.43,2.0,N,132,230,4,-70.0,0.0,-0.5,0.0,-6.94,-1.0,-83.44,-2.5,-1.75,-0.75


In [182]:
data[(data["vendor_id"] == 2)]

,vendor_id,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pu_location_id,do_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.20,1.0,0.5,0.0,0.0,1.0,9.70,0.0,0.0,0.00
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.80,1.0,0.5,0.0,0.0,1.0,8.30,0.0,0.0,0.00
5,2,2025-01-01 00:48:24,2025-01-01 01:08:26,2.0,2.63,1.0,N,239,68,2,19.10,1.0,0.5,0.0,0.0,1.0,24.10,2.5,0.0,0.00
9,2,2025-01-01 00:00:02,2025-01-01 00:09:36,1.0,1.71,1.0,N,237,262,2,11.40,1.0,0.5,0.0,0.0,1.0,16.40,2.5,0.0,0.00
10,2,2025-01-01 00:20:28,2025-01-01 00:28:04,1.0,2.29,1.0,N,237,75,2,11.40,1.0,0.5,0.0,0.0,1.0,16.40,2.5,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3475221,2,2025-01-31 23:01:48,2025-01-31 23:16:29,NaN,3.35,NaN,None,79,237,0,15.85,0.0,0.5,0.0,0.0,1.0,20.60,NaN,NaN,0.75
3475222,2,2025-01-31 23:50:29,2025-02-01 00:17:27,NaN,8.73,NaN,None,161,116,0,28.14,0.0,0.5,0.0,0.0,1.0,32.89,NaN,NaN,0.75
3475223,2,2025-01-31 23:26:59,2025-01-31 23:43:01,NaN,2.64,NaN,None,144,246,0,14.91,0.0,0.5,0.0,0.0,1.0,19.66,NaN,NaN,0.75
3475224,2,2025-01-31 23:14:34,2025-01-31 23:34:52,NaN,3.16,NaN,None,142,107,0,17.55,0.0,0.5,0.0,0.0,1.0,22.30,NaN,NaN,0.75


In [208]:
data[data['tpep_pickup_datetime'] > data['tpep_dropoff_datetime']]

,vendor_id,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pu_location_id,do_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee
99620,1,2025-01-02 12:26:00,2025-01-02 11:29:58,1.0,9.00,99.0,N,88,210,1,45.5,0.0,0.5,0.0,0.00,0.0,46.00,0.0,0.0,0.0
459885,1,2025-01-06 16:00:00,2025-01-06 15:05:30,1.0,3.80,99.0,N,208,254,1,24.5,0.0,0.5,0.0,0.00,0.0,25.00,0.0,0.0,0.0
1315880,1,2025-01-15 15:00:00,2025-01-15 14:42:48,1.0,1.00,99.0,N,107,4,1,18.5,0.0,0.5,0.0,0.00,0.0,19.00,0.0,0.0,0.0
2029534,1,2025-01-23 01:44:59,2024-12-18 07:52:40,0.0,3.10,1.0,N,162,238,1,21.9,2.5,0.5,4.5,0.00,1.0,30.40,2.5,0.0,0.0
2058555,1,2025-01-23 12:30:00,2025-01-23 11:44:59,1.0,3.90,99.0,N,236,116,1,27.5,0.0,0.5,0.0,0.00,0.0,28.00,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3419203,6,2025-01-29 22:01:23,2025-01-29 22:01:13,NaN,4.92,NaN,None,61,76,0,2.9,0.0,0.5,0.0,0.00,0.3,22.64,NaN,NaN,0.0
3424926,6,2025-01-30 08:01:40,2025-01-30 08:01:09,NaN,13.77,NaN,None,146,175,0,2.9,0.0,0.5,0.0,0.00,0.3,44.72,NaN,NaN,0.0
3430705,6,2025-01-30 16:01:38,2025-01-30 16:01:31,NaN,5.18,NaN,None,142,42,0,2.9,0.0,0.5,0.0,0.00,0.3,27.81,NaN,NaN,0.0
3443840,6,2025-01-30 23:01:38,2025-01-30 23:01:26,NaN,7.30,NaN,None,183,119,0,2.9,0.0,0.5,0.0,0.00,0.3,24.20,NaN,NaN,0.0


In [210]:
# Negativos por tipo de pago y vendor
money_cols = ['fare_amount','extra','mta_tax','tip_amount','tolls_amount',
              'improvement_surcharge','total_amount','congestion_surcharge',
              'airport_fee','cbd_congestion_fee']

for c in money_cols:
    if c in data.columns:
        tmp = data.assign(neg=(data[c] < 0))
        resumen = tmp.groupby(['payment_type','vendor_id'])['neg'].mean().sort_values(ascending=False)
        print(f"\n--- % negativos en {c} ---")
        print(resumen.head(15))


--- % negativos en fare_amount ---
payment_type  vendor_id
3             2            0.526505
4             2            0.524563
0             2            0.187885
2             2            0.045404
1             2            0.000007
0             6            0.000000
              1            0.000000
2             1            0.000000
1             7            0.000000
              1            0.000000
2             7            0.000000
3             1            0.000000
              7            0.000000
4             1            0.000000
5             1            0.000000
Name: neg, dtype: float64

--- % negativos en extra ---
payment_type  vendor_id
4             2            0.264757
3             2            0.241181
2             2            0.021768
0             1            0.004467
1             2            0.000004
0             6            0.000000
              2            0.000000
2             1            0.000000
1             7            0.000

In [ ]:
# Duración y velocidad implícita (reglas físicas)
df = data.copy()
df['duration_min'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60
df['speed_mph'] = df['trip_distance'] / (df['duration_min'] / 60)

print("Duración < 0:", (df['duration_min'] < 0).sum())
print("Duración == 0:", (df['duration_min'] == 0).sum())
print("Velocidad > 80 mph:", (df['speed_mph'] > 80).sum())
print("Velocidad > 120 mph:", (df['speed_mph'] > 120).sum())

Duración < 0: 124
Duración == 0: 1927
Velocidad > 80 mph: 2217
Velocidad > 120 mph: 2073


In [ ]:
df[df["speed_mph"] > 120]

,vendor_id,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pu_location_id,do_location_id,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,duration_min,speed_mph
7872,1,2025-01-01 01:29:00,2025-01-01 01:29:16,1.0,2.20,5.0,N,37,37,1,...,0.0,5.00,0.0,1.0,60.00,0.0,0.0,0.00,0.266667,495.000000
8040,1,2025-01-01 01:13:07,2025-01-01 01:13:33,1.0,3.00,1.0,N,246,246,3,...,0.5,0.00,0.0,1.0,8.00,2.5,0.0,0.00,0.433333,415.384615
12954,1,2025-01-01 02:32:52,2025-01-01 02:33:53,3.0,6.10,5.0,N,48,48,1,...,0.0,6.20,0.0,1.0,37.20,0.0,0.0,0.00,1.016667,360.000000
13736,1,2025-01-01 02:15:12,2025-01-01 02:15:36,1.0,3.40,5.0,N,97,97,1,...,0.0,0.00,0.0,1.0,13.00,0.0,0.0,0.00,0.400000,510.000000
16610,1,2025-01-01 03:32:51,2025-01-01 03:33:45,1.0,4.00,1.0,N,231,231,4,...,0.5,0.00,0.0,1.0,8.00,2.5,0.0,0.00,0.900000,266.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3461154,2,2025-01-31 18:33:00,2025-01-31 18:53:00,NaN,71976.78,NaN,None,41,243,0,...,0.5,0.00,0.0,1.0,29.58,NaN,NaN,0.00,20.000000,215930.340000
3461434,2,2025-01-31 18:13:00,2025-01-31 18:21:00,NaN,88102.25,NaN,None,48,161,0,...,0.5,0.00,0.0,1.0,5.08,NaN,NaN,0.75,8.000000,660766.875000
3464668,2,2025-01-31 19:24:00,2025-01-31 19:35:00,NaN,60677.15,NaN,None,170,113,0,...,0.5,0.00,0.0,1.0,14.91,NaN,NaN,0.75,11.000000,330966.272727
3468302,6,2025-01-31 21:01:13,2025-01-31 21:01:44,NaN,7.55,NaN,None,137,167,0,...,0.5,0.00,0.0,0.3,25.16,NaN,NaN,0.00,0.516667,876.774194


In [223]:
df["speed_mph"].describe()

c:\Users\nicoc\AppData\Local\Programs\Python\Python310\lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


count    3.474534e+06
mean              inf
std               NaN
min     -6.436800e+04
25%      7.209809e+00
50%      9.519231e+00
75%      1.293839e+01
max               inf
Name: speed_mph, dtype: float64

In [235]:
# contar duplicados

duplicate_count = data.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 0


In [224]:
data.columns

Index(['vendor_id', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'rate_code_id',
       'store_and_fwd_flag', 'pu_location_id', 'do_location_id',
       'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount',
       'tolls_amount', 'improvement_surcharge', 'total_amount',
       'congestion_surcharge', 'airport_fee', 'cbd_congestion_fee'],
      dtype='object')

In [279]:
from typing import Dict

def transform(data, *args, **kwargs) -> Dict[str, pd.DataFrame]:
    # Tipado
    for col in ['tpep_pickup_datetime', 'tpep_dropoff_datetime']:
        if col in data.columns:
            data[col] = pd.to_datetime(data[col], errors='coerce')

    numeric_cols = [
        'vendor_id', 'rate_code_id', 'pu_location_id', 'do_location_id',
        'payment_type', 'passenger_count', 'trip_distance', 'fare_amount',
        'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
        'improvement_surcharge', 'congestion_surcharge', 'airport_fee', 'total_amount'
    ]
    for col in numeric_cols:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors='coerce')

    # Limpieza
    data = data.drop_duplicates()

    data['rate_code_id'] = data['rate_code_id'].fillna(99).astype(int)

    data.rename(columns={'payment_type': 'payment_type_id'}, inplace=True)

    data['trip_duration_minutes'] = (
        data['tpep_dropoff_datetime'] - data['tpep_pickup_datetime']
    ).dt.total_seconds() / 60.0

    data['avg_speed_mph'] = data['trip_distance'] / (data['trip_duration_minutes'] / 60.0)

    # Facts
    fact = data.copy()

    # Dimensions (https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf)
    dim_vendor = data[['vendor_id']].drop_duplicates().sort_values('vendor_id').reset_index(drop=True)
    dim_vendor['vendor_name'] = dim_vendor['vendor_id'].map({
        1: 'Creative Mobile Technologies, LLC',
        2: 'Curb Mobility, LLC',
        6: 'Myle Technologies Inc',
        7: 'Helix'
    }).fillna('Unknown Vendor')

    dim_payment_type = data[['payment_type_id']].drop_duplicates().sort_values('payment_type_id').reset_index(drop=True)
    dim_payment_type['payment_type_name'] = dim_payment_type['payment_type_id'].map({
        0: 'Flex Fare trip',
        1: 'Credit card',
        2: 'Cash',
        3: 'No charge',
        4: 'Dispute',
        5: 'Unknown',
        6: 'Voided trip'
    }).fillna('Unknown Payment Type')

    dim_pickup_location = data[['pu_location_id']].drop_duplicates().sort_values('pu_location_id').reset_index(drop=True)
    dim_dropoff_location = data[['do_location_id']].drop_duplicates().sort_values('do_location_id').reset_index(drop=True)

    dim_rate_code = data[['rate_code_id']].drop_duplicates().sort_values('rate_code_id').reset_index(drop=True)
    dim_rate_code['rate_code_name'] = dim_rate_code['rate_code_id'].map({
        1: 'Standard rate',
        2: 'JFK',
        3: 'Newark',
        4: 'Nassau or Westchester',
        5: 'Negotiated fare',
        6: 'Group ride',
        99: 'Null/unknown'
    }).fillna('Null/unknown')

    return {
        'dim_vendor': dim_vendor,
        'dim_payment_type': dim_payment_type,
        'dim_pickup_location': dim_pickup_location,
        'dim_dropoff_location': dim_dropoff_location,
        'dim_rate_code': dim_rate_code,
        'fact_taxi_trip': fact,
    }

# test
df = transform(data)
for name, table in df.items():
    print(f"{name}: {table.shape[0]} rows, {table.shape[1]} columns")
    display(table.head())

dim_vendor: 4 rows, 2 columns


,vendor_id,vendor_name
0,1,"Creative Mobile Technologies, LLC"
1,2,"Curb Mobility, LLC"
2,6,Myle Technologies Inc
3,7,Helix


dim_payment_type: 6 rows, 2 columns


,payment_type_id,payment_type_name
0,0,Flex Fare trip
1,1,Credit card
2,2,Cash
3,3,No charge
4,4,Dispute


dim_pickup_location: 261 rows, 1 columns


...

,do_location_id
0,1
1,2
2,3
3,4
4,5


dim_rate_code: 7 rows, 2 columns


,rate_code_id,rate_code_name
0,1,Standard rate
1,2,JFK
2,3,Newark
3,4,Nassau or Westchester
4,5,Negotiated fare


fact_taxi_trip: 3475226 rows, 22 columns


,vendor_id,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pu_location_id,do_location_id,payment_type_id,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,trip_duration_minutes,avg_speed_mph
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1,N,229,237,1,...,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0,8.350000,11.497006
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1,N,236,237,1,...,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0,2.550000,11.764706
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1,N,141,141,1,...,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0,1.950000,18.461538
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1,N,244,244,2,...,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0,5.566667,5.604790
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1,N,244,116,2,...,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0,3.533333,11.207547


In [280]:
def _format_in_clause(values):
    # Convierte una lista de valores en texto SQL para IN (...)
    formatted = []
    for v in values:
        if pd.isna(v):
            continue
        if isinstance(v, (int, float)) and float(v).is_integer():
            formatted.append(str(int(v)))
        else:
            escaped = str(v).replace("'", "''")
            formatted.append(f"'{escaped}'")
    return ", ".join(formatted)

def export_clean_data(model: Dict[str, pd.DataFrame], **kwargs) -> None:
    dim_specs = [
        ('dim_vendor', 'vendor_id'),
        ('dim_payment_type', 'payment_type_id'),
        ('dim_pickup_location', 'pu_location_id'),
        ('dim_dropoff_location', 'do_location_id'),
    ]

    for table_name, key_col in dim_specs:
        df = model[table_name].copy()
        keys = df[key_col].dropna().drop_duplicates().tolist()

        if keys:
            delete_query = f"""
                DELETE FROM clean.{table_name}
                WHERE {key_col} IN ({_format_in_clause(keys)});
            """
            print(f"Executing query for {table_name}:\n{delete_query}\n")

# test export
export_clean_data(df)

Executing query for dim_vendor:

                DELETE FROM clean.dim_vendor
                WHERE vendor_id IN (1, 2, 6, 7);
            

Executing query for dim_payment_type:

                DELETE FROM clean.dim_payment_type
                WHERE payment_type_id IN (0, 1, 2, 3, 4, 5);
            

Executing query for dim_pickup_location:

                DELETE FROM clean.dim_pickup_location
                WHERE pu_location_id IN (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 106, 107, 108, 109, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 

In [ ]:
def export_clean_to_postgres(data: DataFrame, **kwargs) -> None:
    schema_name = "clean"
    table_name = kwargs.get("table_name", "ny_taxi_trips_clean")
    config_path = path.join(get_repo_path(), "io_config.yaml")
    config_profile = kwargs.get("config_profile", "default")
    chunksize = int(kwargs.get("chunksize", 500000))

    execution_date = _parse_execution_date(kwargs.get("execution_date"))
    current_year = int(execution_date.year)
    current_month = int(execution_date.month)

    if data is None or data.empty:
        print(f"[INFO] No rows to export for {current_year}-{current_month:02d}")
        return

    df = data.copy()

    # Garantiza columnas de partición para DELETE idempotente por mes.
    if "trip_year" not in df.columns or "trip_month" not in df.columns:
        if "tpep_pickup_datetime" not in df.columns:
            raise ValueError(
                "Data must include (trip_year, trip_month) or tpep_pickup_datetime."
            )

        df["tpep_pickup_datetime"] = pd.to_datetime(
            df["tpep_pickup_datetime"], errors="coerce"
        )
        df["trip_year"] = df["tpep_pickup_datetime"].dt.year
        df["trip_month"] = df["tpep_pickup_datetime"].dt.month

    # Exporta solo la partición objetivo (clave para backfill controlado).
    df = df[
        (df["trip_year"] == current_year) &
        (df["trip_month"] == current_month)
    ].copy()

    rows = df.shape[0]
    if rows == 0:
        print(f"[INFO] Partition {current_year}-{current_month:02d} has 0 rows")
        return

    with Postgres.with_config(ConfigFileLoader(config_path, config_profile)) as loader:
        print(f"[INFO] Exporting {rows} rows to {schema_name}.{table_name} for {current_year}-{current_month:02d}")

        delete_query = f"""
            DELETE FROM {schema_name}.{table_name}
            WHERE trip_year = {current_year}
              AND trip_month = {current_month};
        """

        try:
            loader.execute(delete_query)
            table_ready = True
            print(f"[OK] Deleted existing partition {current_year}-{current_month:02d} (if any)")
        except Exception as exc:
            loader.conn.rollback()
            table_ready = False
            print(
                f"[WARN] Delete step skipped for {current_year}-{current_month:02d}: {exc}. "
                "Will rebuild/append table data instead."
            )

        for i in range(0, rows, chunksize):
            chunk = df.iloc[i:i + chunksize]

            loader.export(
                chunk,
                schema_name,
                table_name,
                index=False,
                if_exists="append" if table_ready or i > 0 else "replace",
            )

            print(
                f"[OK] Exported chunk {i // chunksize + 1} "
                f"({chunk.shape[0]} rows) for {current_year}-{current_month:02d}"
            )

        print(f"[OK] Loaded partition {current_year}-{current_month:02d} ({rows} rows)")

In [ ]:
from mage_ai.settings.repo import get_repo_path
from mage_ai.io.config import ConfigFileLoader
from mage_ai.io.postgres import Postgres
from pandas import DataFrame
from os import path

if 'data_exporter' not in globals():
    from mage_ai.data_preparation.decorators import data_exporter


# --------------------------
# Exporter para facts
# --------------------------
@data_exporter
def export_facts_to_postgres(output: dict, **kwargs) -> None:
    schema_name = 'clean'
    table_name = 'fact_taxi_trip'
    data = output['facts'][table_name]  # Solo tomamos los facts
    config_path = path.join(get_repo_path(), 'io_config.yaml')
    config_profile = 'default'

    execution_date = kwargs.get('execution_date')
    if execution_date is None:
        raise ValueError("Failed to get execution_date")

    current_year = int(execution_date.year)
    current_month = int(execution_date.month)

    chunksize = int(kwargs.get("chunksize", 500000))
    rows = data.shape[0]

    with Postgres.with_config(ConfigFileLoader(config_path, config_profile)) as loader:
        print(f"[INFO] Exporting facts for {current_year}-{current_month:02d} to PostgreSQL...")

        # Elimina solo los registros del mismo mes/año
        delete_query = f"""
            DELETE FROM {schema_name}.{table_name}
            WHERE trip_year = {current_year}
              AND trip_month = {current_month};
        """
        try:
            loader.execute(delete_query)
            table_ready = True
            print(f"[OK] Deleted existing records for {current_year}-{current_month:02d}")
        except Exception as exc:
            loader.conn.rollback()
            print(
                f"[WARN] Delete step skipped for {current_year}-{current_month:02d}: {exc}. "
                "Will append data instead."
            )
            table_ready = False

        for i in range(0, rows, chunksize):
            chunk = data.iloc[i:i + chunksize]

            loader.export(
                chunk,
                schema_name,
                table_name,
                index=False,
                if_exists='append' if table_ready or i > 0 else 'replace',
            )
            
            print(f"[OK] Exported chunk {i // chunksize + 1} ({chunk.shape[0]} rows)")

        print(f"[OK] Loaded {rows} rows into {schema_name}.{table_name}")


# --------------------------
# Exporter para dimensiones
# --------------------------
@data_exporter
def export_dimensions_to_postgres(output: dict, **kwargs) -> None:
    dims = output['dimensions']  # Solo tomamos las dimensiones
    schema_name = 'clean'
    config_path = path.join(get_repo_path(), 'io_config.yaml')
    config_profile = 'default'

    with Postgres.with_config(ConfigFileLoader(config_path, config_profile)) as loader:
        for dim_name, df_dim in dims.items():
            print(f"[INFO] Exporting dimension {dim_name} ({df_dim.shape[0]} rows) to PostgreSQL...")
            try:
                # Reemplaza toda la tabla
                loader.export(
                    df_dim,
                    schema_name,
                    dim_name,
                    index=False,
                    if_exists='replace',
                )
                print(f"[OK] Dimension {dim_name} exported successfully")
            except Exception as exc:
                loader.conn.rollback()
                print(f"[ERROR] Failed to export dimension {dim_name}: {exc}")